# Assessment Notebook

In [1]:
# Numerical arrays
import numpy as np

#  Yahoo Finance data
import yfinance as yf

# Dates and times
import datetime as dt

# Data frames
import pandas as pd

# Plotting
import matplotlib.pyplot as plt

# File system paths
from pathlib import Path

## Problem 1: Downloading Stock Market Data

Problem 1 involves downloading five days of hourly stock market data for five companies: NVIDIA (NVDA), Broadcom (AVGO), Arm (ARM), AMD (AMD), IBM (IBM).

I use the [yfinance](https://github.com/ranaroussi/yfinance) package, it retrieves data from Yahoo Finance. The function `get_data()` does four things:

1. Downloads the data for all five tickers in a single call to `yf.download()`, using `period="5d"` and `interval="1h"`.
2. Creates a `data` folder in the root of repository if it does not already exist.
3. Saves the data as a CSV file named with the download time in the format `YYYYMMDD-HHmmss.csv`, so each run creates a new file and nothing is overwritten.
4. Returns the downloaded data as a pandas DataFrame so it can be used directly.

In [2]:
def get_data():
    # Ticker symbols for the five companies
    tickers = ["NVDA", "AVGO", "ARM", "AMD", "IBM"]

    # One call for all tickers: faster than five separate downloads and gives a single table with the same timestamps for every company
    data = yf.download(tickers, period="5d", interval="1h", progress=False)

    # exist_ok=True so the function does not fail if the folder exists
    data_folder = Path("data")
    data_folder.mkdir(exist_ok=True)

    # Timestamp in the filename means each run creates a new file
    timestamp = dt.datetime.now().strftime("%Y%m%d-%H%M%S")
    filename = data_folder / f"{timestamp}.csv"

    # Save the data, including the multi-level column headers
    data.to_csv(filename)

    return data

Calling the function downloads the data, saves the CSV and returns the DataFrame. `head()` shows the first five rows.

In [3]:
df = get_data()
df.head()

Price                           Close                                      \
Ticker                            AMD         ARM        AVGO         IBM   
Datetime                                                                    
2026-10-28 09:30:00-04:00  563.799988  233.335007  346.640015  251.039993   
2026-10-28 10:30:00-04:00  566.080017  232.610001  345.795013  249.800003   
2026-10-28 11:30:00-04:00  567.094971  232.339996  347.119995  251.455002   
2026-10-28 12:30:00-04:00  567.979980  231.979996  347.911987  252.509995   
2026-10-28 13:30:00-04:00  571.780029  232.630005  349.059998  252.160004   

Price                                        High                          \
Ticker                           NVDA         AMD         ARM        AVGO   
Datetime                                                                    
2026-10-28 09:30:00-04:00  218.250107  573.960022  233.449997  351.609985   
2026-10-28 10:30:00-04:00  217.630005  568.759888  233.764496  346.899994   
2026-10-28 11:30:00-04:00  217.080002  568.109985  233.199997  347.409912   
2026-10-28 12:30:00-04:00  216.764999  571.898987  234.160004  348.892792   
2026-10-28 13:30:00-04:00  217.039993  571.929993  232.990005  349.244995   

Price                                              ...        Open  \
Ticker                            IBM        NVDA  ...         AMD   
Datetime                                           ...               
2026-10-28 09:30:00-04:00  252.299896  219.990005  ...  572.520020   
2026-10-28 10:30:00-04:00  251.570007  218.449997  ...  563.570007   
2026-10-28 11:30:00-04:00  251.748993  217.820007  ...  566.200012   
2026-10-28 12:30:00-04:00  253.850006  217.750000  ...  567.109985   
2026-10-28 13:30:00-04:00  252.570007  217.539993  ...  567.950012   

Price                                                                      \
Ticker                            ARM        AVGO         IBM        NVDA   
Datetime                                                                    
2026-10-28 09:30:00-04:00  231.000000  348.720001  248.490005  219.880005   
2026-10-28 10:30:00-04:00  233.335007  346.640015  251.080002  218.240005   
2026-10-28 11:30:00-04:00  232.705002  345.779999  249.740005  217.634995   
2026-10-28 12:30:00-04:00  232.429993  347.174988  251.500000  217.085007   
2026-10-28 13:30:00-04:00  232.000000  347.920013  252.470001  216.755005   

Price                       Volume                                     
Ticker                         AMD     ARM     AVGO     IBM      NVDA  
Datetime                                                               
2026-10-28 09:30:00-04:00  4303815  901655  4640300  643364  17169156  
2026-10-28 10:30:00-04:00  1865142  380795  1598952  268683   6987823  
2026-10-28 11:30:00-04:00  1212023  226612  1359851  204691   7855021  
2026-10-28 12:30:00-04:00  1508549  291888  1297880  368351   5871987  
2026-10-28 13:30:00-04:00  1417467  229665  1333282  297096   6207460  

[5 rows x 25 columns]

### References
- [yfinance documentation](https://ranaroussi.github.io/yfinance): describes `yf.download()` and its `period` and `interval` arguments, which I use to request five days of hourly data for several tickers at once.
- [Python `pathlib` documentation](https://docs.python.org/3/library/pathlib.html): explains `Path.mkdir(exist_ok=True)`, which I use to create the `data` folder only when it is missing.
- [Python `datetime` documentation: strftime format codes](https://docs.python.org/3/library/datetime.html#strftime-and-strptime-format-codes): explains the codes `%Y%m%d-%H%M%S` that I use to build the timestamp for the filename.

## Problem 2: Plotting the Data

Problem 2 involves plotting the stock data downloaded in Problem  1. The  function `plot_data()`will:

1. Open the most recent CSV file in the `data` folder.
2. Calculate the percentage change in the `Close` price of each company, relative to its first recorded observation.
3. Plot the percentage change for all five companies on the same set of axes.
4. Save the plot in a `plots` folder, named in the format `YYYYMMDD-HHmmss.png`.

The CSV filenames are timestamps in the format `YYYYMMDD-HHmmss`, so sorting them alphabetically also sorts them by time. The last file in the sorted list is the most recent download. The file has two header rows (price type and ticker), so `header=[0, 1]` tells pandas to read both as column levels.

In [4]:
# Sorting the names puts the newest timestamp last
latest_file = sorted(Path("data").glob("*.csv"))[-1]
print(latest_file)

# Two header rows: price type (Close, High, ...) and ticker
prices = pd.read_csv(latest_file, header=[0, 1], index_col=0, parse_dates=True)
prices["Close"].head()

data\20261103-115537.csv


Ticker,AMD,ARM,AVGO,IBM,NVDA
Datetime,,,,,
2026-10-27 09:30:00-04:00,551.700012,223.750000,339.299988,249.570007,217.119995
2026-10-27 10:30:00-04:00,556.179993,226.259995,341.059998,250.115005,217.793503
2026-10-27 11:30:00-04:00,558.780029,227.289993,342.609985,249.964996,218.654999
2026-10-27 12:30:00-04:00,559.031982,225.990005,342.220001,250.289993,217.554993
2026-10-27 13:30:00-04:00,559.679993,226.779999,342.589996,249.910004,217.524994


## Problem 3: Creating an Executable Script

Problem 3 involves

## Problem 4: Automating the Analysis with GitHub Actions

Problem 4 involves

## End